<a href="https://colab.research.google.com/github/jalisonSantos/Inteligencia-artificial/blob/main/Aula_15_Contagem_Carrosatividade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>Universidade Univille</center>
<center>Unidade 04 - Aprendizado de Máquinas</center>
<center>Classificação de Imagens</center>
<center>TURMA 2026/2</center>

**<center>AULA 15: Contagem de Carros com Visão Computacional</center>**

---
## Objetivo

Desenvolver uma solução com **visão computacional** para **detectar e contar carros** em imagens de ruas, utilizando:
- Haar Cascade Classifier (OpenCV)
- Pré-processamento de imagem
- Contagem automática de veículos detectados
---

## 01. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS

In [ ]:
# Instalação (caso necessário)
!pip install opencv-python-headless matplotlib numpy requests -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
from google.colab.patches import cv2_imshow

print('OpenCV versão:', cv2.__version__)
print('NumPy versão:', np.__version__)

---
## 02. DOWNLOAD DA IMAGEM DE TESTE

Vamos baixar uma imagem de rua com carros para testar nossa solução.

In [ ]:
# Baixar uma imagem de carros em rua (domínio público)
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'

# ⚠️ SUBSTITUA pela URL de uma imagem real de carros em rua,
# ou faça upload manual no Colab com o caminho abaixo:
IMAGE_PATH = '/content/carros_rua.jpg'

# Para upload manual no Colab:
from google.colab import files
print('Faça upload de uma imagem de rua com carros:')
uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]
print(f'Imagem carregada: {IMAGE_PATH}')

---
## 03. CARREGAMENTO E VISUALIZAÇÃO DA IMAGEM

In [ ]:
# Carregar imagem
img_original = cv2.imread(IMAGE_PATH)

# Verificação
if img_original is None:
    raise FileNotFoundError(f'Imagem não encontrada em: {IMAGE_PATH}')

# Informações da imagem
print('Dimensões (linhas, colunas, bandas):', img_original.shape)
print('Total de pixels:', img_original.size)
print('Tipo de dado:', img_original.dtype)

# Exibir imagem original
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
plt.title('Imagem Original', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## 04. PRÉ-PROCESSAMENTO DA IMAGEM

Antes da detecção, aplicamos técnicas de pré-processamento para melhorar a qualidade da análise.

In [ ]:
# --- Pré-processamento ---

# 1. Converter para escala de cinza
gray = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

# 2. Equalização de histograma (melhora contraste)
gray_eq = cv2.equalizeHist(gray)

# 3. Suavização Gaussiana (reduz ruído)
blur = cv2.GaussianBlur(gray_eq, (5, 5), 0)

# Visualizar etapas do pré-processamento
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('1. Escala de Cinza', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(gray_eq, cmap='gray')
axes[1].set_title('2. Equalização de Histograma', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(blur, cmap='gray')
axes[2].set_title('3. Suavização Gaussiana', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Etapas de Pré-processamento', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 05. HISTOGRAMA DA IMAGEM

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma original
axes[0].hist(gray.ravel(), bins=256, color='steelblue', alpha=0.8)
axes[0].set_title('Histograma - Imagem Original (Cinza)')
axes[0].set_xlabel('Intensidade de Pixel')
axes[0].set_ylabel('Quantidade de Pixels')

# Histograma após equalização
axes[1].hist(gray_eq.ravel(), bins=256, color='darkorange', alpha=0.8)
axes[1].set_title('Histograma - Após Equalização')
axes[1].set_xlabel('Intensidade de Pixel')
axes[1].set_ylabel('Quantidade de Pixels')

plt.tight_layout()
plt.show()

---
## 06. DETECÇÃO DE CARROS COM HAAR CASCADE

O **Haar Cascade** é um método clássico de visão computacional treinado para detectar objetos específicos (como rostos, olhos, carros) em imagens.

Parâmetros importantes do `detectMultiScale`:
- `scaleFactor`: quanto a imagem é reduzida a cada escala (ex: 1.1 = 10%)
- `minNeighbors`: quantos vizinhos uma detecção precisa ter para ser válida (reduz falsos positivos)
- `minSize`: tamanho mínimo do objeto detectado

In [ ]:
# Baixar o classificador de carros (cars.xml)
!wget -q https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml -O /content/cars.xml
print('Classificador baixado com sucesso!')

In [ ]:
# Carregar o classificador Haar Cascade para carros
car_cascade = cv2.CascadeClassifier('/content/cars.xml')

# Verificar se carregou corretamente
if car_cascade.empty():
    raise IOError('Erro ao carregar o classificador. Verifique o arquivo cars.xml')
else:
    print('Classificador carregado com sucesso!')

# Detectar carros na imagem pré-processada
carros = car_cascade.detectMultiScale(
    blur,
    scaleFactor=1.1,
    minNeighbors=3,
    minSize=(60, 60)
)

print(f'\nTotal de carros detectados: {len(carros)}')

---
## 07. VISUALIZAÇÃO DO RESULTADO - DETECÇÃO E CONTAGEM

In [ ]:
# Criar cópia da imagem para desenhar as detecções
img_resultado = img_original.copy()

# Contador
contador = 0

# Desenhar retângulos em cada carro detectado
for (x, y, w, h) in carros:
    contador += 1
    # Retângulo verde ao redor do carro
    cv2.rectangle(img_resultado, (x, y), (x + w, y + h), (0, 255, 0), 2)
    # Número do carro acima do retângulo
    cv2.putText(
        img_resultado,
        f'Carro {contador}',
        (x, y - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

# Exibir contagem total no canto superior esquerdo
cv2.rectangle(img_resultado, (5, 5), (280, 45), (0, 0, 0), -1)
cv2.putText(
    img_resultado,
    f'Total de Carros: {contador}',
    (10, 32),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.9,
    (0, 255, 255),
    2
)

# Exibir resultado
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(img_resultado, cv2.COLOR_BGR2RGB))
plt.title(f'Resultado da Detecção — {contador} carro(s) detectado(s)', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'\n✅ Contagem final: {contador} carro(s) detectado(s) na imagem.')

---
## 08. COMPARAÇÃO: ORIGINAL vs DETECÇÃO

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
axes[0].set_title('Imagem Original', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_resultado, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detecção: {contador} carro(s) encontrado(s)', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Comparação: Original vs Detecção de Carros', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 09. AJUSTE DE PARÂMETROS (EXPERIMENTO)

Teste diferentes configurações do `detectMultiScale` e compare os resultados.

In [ ]:
# Configurações para testar
configuracoes = [
    {'scaleFactor': 1.05, 'minNeighbors': 2, 'minSize': (40, 40)},
    {'scaleFactor': 1.1,  'minNeighbors': 3, 'minSize': (60, 60)},
    {'scaleFactor': 1.15, 'minNeighbors': 5, 'minSize': (80, 80)},
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, cfg in enumerate(configuracoes):
    deteccoes = car_cascade.detectMultiScale(
        blur,
        scaleFactor=cfg['scaleFactor'],
        minNeighbors=cfg['minNeighbors'],
        minSize=cfg['minSize']
    )

    img_exp = img_original.copy()
    for (x, y, w, h) in deteccoes:
        cv2.rectangle(img_exp, (x, y), (x + w, y + h), (0, 200, 50), 2)

    axes[i].imshow(cv2.cvtColor(img_exp, cv2.COLOR_BGR2RGB))
    axes[i].set_title(
        f"scale={cfg['scaleFactor']} | neighbors={cfg['minNeighbors']}\n"
        f"minSize={cfg['minSize']} | Detectados: {len(deteccoes)}",
        fontsize=10
    )
    axes[i].axis('off')

plt.suptitle('Comparação de Parâmetros do detectMultiScale', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. SALVAR IMAGEM COM O RESULTADO

In [ ]:
output_path = '/content/resultado_contagem_carros.jpg'
cv2.imwrite(output_path, img_resultado)
print(f'Imagem salva em: {output_path}')

# Download automático (Colab)
from google.colab import files
files.download(output_path)

---
## 11. CONCLUSÃO

### Resumo da Solução:

| Etapa | Técnica Utilizada |
|---|---|
| Carregamento | `cv2.imread()` |
| Pré-processamento | Escala de cinza, Equalização de histograma, Gaussian Blur |
| Detecção | Haar Cascade Classifier (`cars.xml`) |
| Contagem | `len(deteccoes)` + marcação visual |
| Visualização | `matplotlib` + `cv2.rectangle()` + `cv2.putText()` |

### Parâmetros-chave do `detectMultiScale`:
- **`scaleFactor`** menor → mais detalhes, mais lento, mais falsos positivos  
- **`minNeighbors`** maior → menos falsos positivos, pode perder detecções  
- **`minSize`** → filtra objetos menores que o esperado

### Referências:
- https://opencv.org/university/
- https://docs.opencv.org/3.4/db/d28/tutorial_cascade_classifier.html
- https://github.com/andrewssobral/vehicle_detection_haarcascades